In [ ]:
import sys
import os
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pca_transform.py").is_file()
)
sys.path.insert(0, str(ROOT))


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, f1_score
from task_1.src.preprocessing import create_dataset_from_h5 as load_data1
from task_2.src.preprocessing import create_dataset_from_h5 as load_data2
from task_1.src.models import build_model_from_pretrained as load1
from task_2.src.models import build_model_from_pretrained as load2
from task_3.src.models import Task3Model
from pca_transform import NpzPCA


In [ ]:
DATA = Path(os.environ.get(
    "PBL_DATA_ROOT", ROOT.parent / "enzyme-classification-data"
))

folder1 = DATA / "task_1/embeddings_uniprot_exp_proteins_c30_flens2-98_enz"
folder2 = DATA / "task_2/embeddings_uniprot_exp_enzymes_c30_flens2-98_usamp2000_exp"

test_set_task_1 = load_data1(
    folder1 / f"{folder1.name}_test.h5", unsqueeze=False
)
test_set_task_2 = load_data2(
    folder2 / f"{folder2.name}_test85.h5", unsqueeze=False
)


In [ ]:
model = Task3Model(
    load1().cpu().eval(),
    load2().cpu().eval(),
    NpzPCA(ROOT / "checkpoints/pca/task1_pca_reconstructed.npz"),
    NpzPCA(ROOT / "checkpoints/pca/task2_pca_original.npz"),
)


In [ ]:
combined_test_sets = list()

for item in test_set_task_1:
    if item[1] == 0:
        combined_test_sets.append(item)

for item in test_set_task_2:
    combined_test_sets.append(item)

In [ ]:
test_set_keys = [item[2].split(' ')[0] for item in combined_test_sets]
test_set_X = torch.stack([item[0] for item in combined_test_sets])
test_set_y = torch.stack([item[1] for item in combined_test_sets])

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(test_set_X)

In [ ]:
from sklearn.metrics import matthews_corrcoef as mcc, f1_score

f1_score(test_set_y, preds, labels=range(0,8), average='macro')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from matplotlib.colors import LogNorm  # <--- Import this

labels = list(range(0,8))

cm = confusion_matrix(test_set_y, preds, normalize='true')

# 2. THE TRANSFORMATION: Zero out the diagonal
# This removes the correct predictions (the dark blue squares), leaving only the errors.
np.fill_diagonal(cm, 0)

fig, ax = plt.subplots(figsize=(8, 8))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

disp.plot(cmap='Blues', ax=ax, colorbar=False, text_kw={'size': 12}, values_format='.1%')

ax.set_xlabel('Predicted class')
ax.set_ylabel('Actual class')

plt.show()